# Omnilingual Selected Dataset Stats

This notebook summarizes row counts, duration statistics, and transcript length statistics for the datasets under `omnilingual_selected`.


In [1]:
from pathlib import Path
from statistics import mean

import pyarrow as pa

base = Path('/home/MohammadNabulsi/whisper/omnilingual_selected')

rows = []
for ds_path in sorted(p for p in base.iterdir() if p.is_dir()):
    durations = []
    char_lens = []
    word_lens = []
    split_rows = {}
    split_duration = {}

    for arrow_path in sorted(ds_path.glob('data-*.arrow')):
        with pa.memory_map(str(arrow_path), 'r') as source:
            table = pa.ipc.open_stream(source).read_all()

        split = table.column('original_split').to_pylist()
        dur = table.column('duration').to_pylist()
        txt = table.column('raw_text').to_pylist()

        for s, d, t in zip(split, dur, txt):
            split_rows[s] = split_rows.get(s, 0) + 1
            split_duration.setdefault(s, []).append(d)
            durations.append(d)
            char_lens.append(len(t) if t else 0)
            word_lens.append(len(t.split()) if t else 0)

    rows.append(
        {
            'dataset': ds_path.name,
            'rows': sum(split_rows.values()),
            'rows_by_split': split_rows,
            'duration_mean_sec': round(mean(durations), 3),
            'duration_min_sec': round(min(durations), 3),
            'duration_max_sec': round(max(durations), 3),
            'duration_mean_by_split_sec': {k: round(mean(v), 3) for k, v in split_duration.items()},
            'text_chars_mean': round(mean(char_lens), 1),
            'text_chars_min': min(char_lens),
            'text_chars_max': max(char_lens),
            'text_words_mean': round(mean(word_lens), 1),
            'text_words_min': min(word_lens),
            'text_words_max': max(word_lens),
        }
    )

rows


[{'dataset': 'apc_north_levantine_all_splits',
  'rows': 517,
  'rows_by_split': {'train': 323, 'dev': 82, 'test': 112},
  'duration_mean_sec': 69.742,
  'duration_min_sec': 4.292,
  'duration_max_sec': 99.288,
  'duration_mean_by_split_sec': {'train': 69.142,
   'dev': 78.679,
   'test': 64.931},
  'text_chars_mean': 750.8,
  'text_chars_min': 42,
  'text_chars_max': 1292,
  'text_words_mean': 131.4,
  'text_words_min': 7,
  'text_words_max': 222},
 {'dataset': 'other_arabic_dialects',
  'rows': 13812,
  'rows_by_split': {'train': 12620, 'dev': 596, 'test': 596},
  'duration_mean_sec': 24.622,
  'duration_min_sec': 0.707,
  'duration_max_sec': 139.205,
  'duration_mean_by_split_sec': {'train': 24.899,
   'dev': 24.64,
   'test': 18.732},
  'text_chars_mean': 240.7,
  'text_chars_min': 5,
  'text_chars_max': 1791,
  'text_words_mean': 43.4,
  'text_words_min': 1,
  'text_words_max': 336}]